# NER Notebook 4: LLM-Based NER

**BSAN 6200 — Text Mining & Social Media Analytics**

## What You'll Learn
1. **Zero-shot NER** — extract entities with no training data using LLM prompts
2. **Few-shot NER** — improve accuracy by providing a few examples
3. **Convert LLM output to IOB2** — bridge LLM results to standard NER evaluation
4. **Practical workflow** — use LLMs to bootstrap annotation in Label Studio

## Why LLM-Based NER?

| Aspect | Traditional NER | LLM NER |
|--------|----------------|---------|
| Training data needed | 200+ labeled sentences | None (zero-shot) or 2-5 examples (few-shot) |
| Setup time | Hours to days | Minutes |
| New entity types | Requires retraining | Just update the prompt |
| Best for | Production, high throughput | Prototyping, low-resource domains, pre-annotation |

## Prerequisites
- Notebooks 1-3 (NER inference, training, pipeline)
- Understanding of IOB2 format and Label Studio

---

---
## Part 1: Zero-Shot NER with LLMs

**The big idea:** Instead of training a model, just *describe* your entities
to an LLM and ask it to extract them. No labeled data needed!

**When to use zero-shot NER:**
- You have NO training data for your entity types
- You need to prototype quickly (test entity definitions in minutes)
- Your entity types are unique to your domain (e.g., "BUSINESS_METRIC", "REGULATION")
- You want to pre-annotate data before human review in Label Studio

In [ ]:
# ============================================================
# SETUP: Load a local instruction-following model
# ============================================================
# We use a local HuggingFace model so you don't need an API key.
# For better results, use GPT-4, Claude, or Gemini via API.

!pip install transformers torch

from transformers import pipeline
import json
import re

# Load a small instruction-following model
# SmolLM2-1.7B-Instruct is small enough for Colab free tier
generator = pipeline(
    "text-generation",
    model="HuggingFaceTB/SmolLM2-1.7B-Instruct",
    max_new_tokens=500,
    device_map="auto",
)
print("Model loaded! Ready for zero-shot NER.")

In [ ]:
# ============================================================
# ZERO-SHOT NER: Define entities in natural language
# ============================================================
# The key is writing a clear, specific prompt that tells the LLM:
# 1. What entity types to extract
# 2. What format to return (JSON for easy parsing)
# 3. What the text is

def zero_shot_ner(text, entity_types, generator):
    """
    Extract named entities using an LLM with zero training data.

    Args:
        text: The input text to analyze
        entity_types: Dict of {entity_name: description}
            Example: {"COMPANY": "Company or organization names"}
        generator: HuggingFace text generation pipeline

    Returns:
        List of dicts with 'text' and 'label' keys, or raw response on failure

    How it works:
        1. Build a prompt describing the entity types
        2. Send to LLM with instruction to return JSON
        3. Parse the JSON response
    """
    # Build the entity type descriptions for the prompt
    entity_desc = "\n".join([f"- {name}: {desc}"
                             for name, desc in entity_types.items()])

    prompt = f"""Extract named entities from the text below.
For each entity, return a JSON list with "text" and "label" keys.
If no entities found, return [].

Entity types to extract:
{entity_desc}

Text: {text}

Return ONLY valid JSON, nothing else:"""

    # Send to LLM
    messages = [{"role": "user", "content": prompt}]
    result = generator(messages)
    response = result[0]["generated_text"][-1]["content"]

    # Try to parse JSON from the response
    json_match = re.search(r'\[.*\]', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass

    # If JSON parsing fails, return the raw response for inspection
    return {"raw_response": response}

In [ ]:
# ============================================================
# DEFINE BUSINESS ENTITY TYPES
# ============================================================
# This is where zero-shot NER shines: you can define ANY entity
# types relevant to YOUR domain, without any training data!

business_entities = {
    "COMPANY": "Company or organization names (e.g., Apple, Goldman Sachs, OpenAI)",
    "PRODUCT": "Product or service names (e.g., iPhone, ChatGPT, S&P 500, Azure)",
    "PERSON": "Person names (e.g., Tim Cook, Warren Buffett, Elon Musk)",
    "MONEY": "Monetary amounts (e.g., $1.5 billion, €500 million, $81.8B)",
    "METRIC": "Business metrics or KPIs (e.g., revenue, EBITDA, market cap, MAU)",
    "DATE": "Dates or time periods (e.g., Q3 2024, fiscal year 2023, last Tuesday)",
}

# Test texts — business/finance domain
test_texts = [
    "Apple reported Q3 revenue of $81.8 billion, beating analyst estimates.",
    "Tesla CEO Elon Musk announced Cybertruck production will ramp up in Austin by March 2025.",
    "Goldman Sachs upgraded Microsoft to buy with a price target of $450, citing Azure growth.",
    "Shopify's gross merchandise volume reached $61 billion in 2023, up 23% year-over-year.",
]

# Run zero-shot NER
print("ZERO-SHOT NER RESULTS")
print("=" * 60)
for text in test_texts:
    print(f"\nText: {text}")
    entities = zero_shot_ner(text, business_entities, generator)
    if isinstance(entities, list):
        for ent in entities:
            if isinstance(ent, dict) and "text" in ent:
                print(f"  → {ent.get('text', '?'):<25} {ent.get('label', '?')}")
    else:
        print(f"  → Could not parse: {str(entities)[:100]}")

---
## Part 2: Few-Shot NER

Few-shot = give the LLM 2-3 examples to follow.
This significantly improves consistency and accuracy compared to zero-shot.

In [ ]:
# ============================================================
# FEW-SHOT NER: Provide examples for better accuracy
# ============================================================
# Few-shot = give the LLM 2-3 examples to follow.
# This significantly improves consistency and accuracy.

def few_shot_ner(text, entity_types, examples, generator):
    """
    Few-shot NER: provide examples for the LLM to follow.

    Args:
        text: Input text to analyze
        entity_types: Dict of entity definitions
        examples: List of (text, json_output_string) tuples as demonstrations
        generator: HuggingFace pipeline

    Returns:
        Parsed entity list or raw response
    """
    entity_desc = "\n".join([f"- {name}: {desc}"
                             for name, desc in entity_types.items()])

    # Build the examples section of the prompt
    examples_text = ""
    for ex_text, ex_output in examples:
        examples_text += f"\nText: {ex_text}\nEntities: {ex_output}\n"

    prompt = f"""You are an NER system. Extract entities and return JSON.

Entity types:
{entity_desc}

Examples:{examples_text}
Now extract entities from this new text. Return ONLY a JSON list:

Text: {text}
Entities:"""

    messages = [{"role": "user", "content": prompt}]
    result = generator(messages)
    response = result[0]["generated_text"][-1]["content"]

    json_match = re.search(r'\[.*\]', response, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except json.JSONDecodeError:
            pass
    return {"raw_response": response}

# Define few-shot examples (these teach the LLM the expected format)
examples = [
    (
        "Amazon Web Services generated $23.1 billion in Q2 2024 revenue.",
        '[{"text": "Amazon Web Services", "label": "COMPANY"}, '
        '{"text": "$23.1 billion", "label": "MONEY"}, '
        '{"text": "Q2 2024", "label": "DATE"}, '
        '{"text": "revenue", "label": "METRIC"}]'
    ),
    (
        "Satya Nadella confirmed Microsoft Teams reached 320 million monthly active users.",
        '[{"text": "Satya Nadella", "label": "PERSON"}, '
        '{"text": "Microsoft Teams", "label": "PRODUCT"}, '
        '{"text": "320 million monthly active users", "label": "METRIC"}]'
    ),
]

# Test with few-shot
test_text = "Nvidia CEO Jensen Huang revealed that H100 GPU sales drove $13.5 billion in data center revenue for Q4 2024."
print("FEW-SHOT NER RESULT")
print("=" * 60)
print(f"Text: {test_text}")
result = few_shot_ner(test_text, business_entities, examples, generator)
if isinstance(result, list):
    for ent in result:
        if isinstance(ent, dict):
            print(f"  → {ent.get('text', '?'):<30} {ent.get('label', '?')}")
else:
    print(f"  → {result}")

---
## Part 3: Converting LLM Output to IOB2

To evaluate LLM NER against ground truth, or to feed LLM outputs into
the training pipeline (Notebook 3), we convert to IOB2 tags.

In [ ]:
# ============================================================
# CONVERT LLM OUTPUT TO IOB2 FORMAT
# ============================================================
# To evaluate LLM NER against ground truth, or to use LLM outputs
# for training, we need to convert to IOB2 tags.

def llm_entities_to_iob(text, entities):
    """
    Convert LLM-extracted entities to IOB2 format.

    This bridges LLM output (JSON with text spans) to the standard
    NER format (one tag per token) used by seqeval and training pipelines.

    Args:
        text: Original text string
        entities: List of dicts with 'text' and 'label' keys

    Returns:
        tokens: List of word tokens
        tags: List of IOB2 tags
    """
    tokens = text.split()
    tags = ["O"] * len(tokens)

    if not isinstance(entities, list):
        return tokens, tags

    for ent in entities:
        if not isinstance(ent, dict) or "text" not in ent or "label" not in ent:
            continue

        ent_tokens = ent["text"].split()
        ent_label = ent["label"]

        # Find the entity span in the token list
        for i in range(len(tokens) - len(ent_tokens) + 1):
            match = all(
                tokens[i + j].strip(".,;:!?()\"\'$").lower() ==
                et.strip(".,;:!?()\"\'$").lower()
                for j, et in enumerate(ent_tokens)
            )
            if match:
                tags[i] = f"B-{ent_label}"
                for j in range(1, len(ent_tokens)):
                    tags[i + j] = f"I-{ent_label}"
                break

    return tokens, tags

# Demo the conversion
demo_text = "Apple reported Q3 revenue of $81.8 billion."
demo_entities = [
    {"text": "Apple", "label": "COMPANY"},
    {"text": "$81.8 billion", "label": "MONEY"},
    {"text": "Q3", "label": "DATE"},
    {"text": "revenue", "label": "METRIC"},
]

tokens, tags = llm_entities_to_iob(demo_text, demo_entities)
print(f"{'Token':<15} {'IOB2 Tag'}")
print("-" * 30)
for tok, tag in zip(tokens, tags):
    marker = " ←" if tag != "O" else ""
    print(f"{tok:<15} {tag}{marker}")

---
## Part 4: Export LLM Predictions to Label Studio

The most powerful use of LLM NER: generate pre-annotations for
human review. This dramatically speeds up the annotation process.

In [ ]:
# ============================================================
# EXPORT LLM PREDICTIONS AS LABEL STUDIO JSON
# ============================================================
# Convert LLM entity extractions to Label Studio's JSON format
# for import as pre-annotations (predictions).

def llm_results_to_label_studio(texts, entity_types, generator, output_file, use_few_shot=False, examples=None):
    """
    Run LLM NER on texts and export as Label Studio JSON.

    Pre-annotations go into the "predictions" field so Label Studio
    shows them as editable suggestions for human reviewers.

    Args:
        texts: list of raw text strings
        entity_types: dict of entity definitions for the prompt
        generator: HuggingFace pipeline
        output_file: path to write JSON
        use_few_shot: whether to use few-shot (requires examples)
        examples: list of (text, json_string) tuples for few-shot
    """
    tasks = []

    for text in texts:
        # Get LLM predictions
        if use_few_shot and examples:
            entities = few_shot_ner(text, entity_types, examples, generator)
        else:
            entities = zero_shot_ner(text, entity_types, generator)

        # Convert to Label Studio format
        results = []
        if isinstance(entities, list):
            for ent in entities:
                if isinstance(ent, dict) and "text" in ent and "label" in ent:
                    # Find the entity span in the original text
                    ent_text = ent["text"]
                    start = text.find(ent_text)
                    if start >= 0:
                        results.append({
                            "from_name": "label",
                            "to_name": "text",
                            "type": "labels",
                            "value": {
                                "start": start,
                                "end": start + len(ent_text),
                                "text": ent_text,
                                "labels": [ent["label"]],
                            },
                        })

        task = {
            "data": {"text": text},
            "predictions": [{"result": results}] if results else [],
        }
        tasks.append(task)

    with open(output_file, "w") as f:
        json.dump(tasks, f, indent=2)

    print(f"Exported {len(tasks)} LLM-annotated tasks to {output_file}")
    print("Import into Label Studio for human review!")

# Example: pre-annotate new business texts
new_texts = [
    "Nvidia CEO Jensen Huang revealed that H100 GPU sales drove $13.5 billion in data center revenue.",
    "Stripe processed $1 trillion in payments in 2023, making it the most valuable fintech startup.",
    "The European Central Bank raised interest rates by 25 basis points in September.",
    "Anthropic released Claude 3, competing with OpenAI's GPT-4 and Google's Gemini.",
]

llm_results_to_label_studio(
    new_texts, business_entities, generator,
    "llm_predictions_for_label_studio.json"
)

# Show a preview
with open("llm_predictions_for_label_studio.json") as f:
    data = json.load(f)
    print(f"\nPreview (first task):")
    print(json.dumps(data[0], indent=2)[:500])

---
## Summary: When to Use LLM NER

| Scenario | Approach | Next Step |
|----------|----------|-----------|
| Exploring new entity types | Zero-shot | Validate definitions, then annotate |
| Small pilot (< 50 texts) | Few-shot | Review quality, refine prompts |
| Bootstrapping annotations | LLM → Label Studio | Human review → train model (NB3) |
| Production NER | Fine-tuned Transformer (NB2) | LLM is too slow/expensive at scale |

### The LLM → Label Studio → Fine-Tune Workflow
1. **Define entities** in natural language (your annotation guideline)
2. **Run LLM NER** (zero-shot or few-shot) on your corpus
3. **Export to Label Studio** as pre-annotations
4. **Human review** — correct LLM mistakes (much faster than annotating from scratch)
5. **Export reviewed data** from Label Studio
6. **Train a model** using Notebook 3 pipeline (CRF or Transformer from NB2)
7. **Iterate** — use the trained model for the next round of pre-annotations